# 모델 평가 및 결과 시각화

- mAP, Precision, Recall 평가
- 클래스별 AP 분석
- 바운딩 박스 탐지 결과 시각화
- 오탐지 (FP) / 미탐지 (FN) 분석

In [23]:
import cv2
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO

print('[OK] 임포트 완료')

[OK] 임포트 완료


### 설정

In [28]:
BASE = Path(r"C:\workspace_python\deeplearning\project")

BEST_WEIGHTS = str(BASE / "runs/detect/pcb_defect_detection/yolov8s_transfer/weights/best.pt")
DATASET_YAML = str(BASE / "pcb_yolo_dataset/dataset.yaml")
TEST_IMG_DIR = BASE / "pcb_yolo_dataset/test/images"
TEST_LBL_DIR = BASE / "pcb_yolo_dataset/test/labels"

CLASSES = [
    "missing_hole", "mouse_bite", "open_circuit",
    "short", "spur", "spurious_copper"
]
COLOR_MAP = {
    "missing_hole"   : "#E63946",
    "mouse_bite"     : "#457B9D",
    "open_circuit"   : "#2A9D8F",
    "short"          : "#E9C46A",
    "spur"           : "#F4A261",
    "spurious_copper": "#264653",
}
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

print(f'best.pt 존재:  {Path(BEST_WEIGHTS).exists()}')
print(f'dataset.yaml:  {Path(DATASET_YAML).exists()}')
print(f'test/images:   {TEST_IMG_DIR.exists()}')
print(f'test/labels:   {TEST_LBL_DIR.exists()}')

best.pt 존재:  True
dataset.yaml:  True
test/images:   True
test/labels:   True


### 모델 로드 및 Test Set 평가

In [29]:
model = YOLO(BEST_WEIGHTS)

print('[EVAL] Test set 평가 시작...')
metrics = model.val(
    data=DATASET_YAML,
    split="test",
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    verbose=True
)

print("\n" + "="*50)
print("         최종 평가 결과 (Test Set)")
print("="*50)
print(f"  mAP@0.5        : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95   : {metrics.box.map:.4f}")
print(f"  Precision      : {metrics.box.mp:.4f}")
print(f"  Recall         : {metrics.box.mr:.4f}")
print("="*50)

[EVAL] Test set 평가 시작...
Ultralytics 8.4.33  Python-3.11.15 torch-2.12.0.dev20260324+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 1132.287.2 MB/s, size: 1651.6 KB)
val: Scanning C:\workspace_python\deeplearning\project\pcb_yolo_dataset\test\labels.cache... 70 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 70/70  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.7s0.3ss
                   all         70        301      0.609      0.241      0.419      0.138
          missing_hole         17         75        0.6       0.28      0.431      0.177
            mouse_bite         11         52      0.571      0.154      0.342      0.109
          open_circuit          9         37      0.455      0.135      0.263     0.0654
                 short         11         39  

In [30]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"C:\workspace_python\deeplearning\project\runs\detect\pcb_defect_detection\yolov8s_v3\results.csv")
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

print(f"실제 학습된 에폭 수: {len(df)}")
print(df[["epoch", "metrics/mAP50(B)", "train/box_loss"]].to_string())

실제 학습된 에폭 수: 50
    epoch  metrics/mAP50(B)  train/box_loss
0       1           0.00000         4.31489
1       2           0.00013         3.92005
2       3           0.00028         3.86956
3       4           0.00391         3.84245
4       5           0.00762         3.92289
5       6           0.01076         3.82119
6       7           0.01870         3.87514
7       8           0.02423         3.77709
8       9           0.01305         3.76035
9      10           0.02250         3.73134
10     11           0.02293         3.75241
11     12           0.03109         3.75764
12     13           0.02123         3.75217
13     14           0.02687         3.73378
14     15           0.02000         3.71481
15     16           0.02416         3.66228
16     17           0.02322         3.68885
17     18           0.01739         3.62024
18     19           0.02412         3.62381
19     20           0.02297         3.66339
20     21           0.02755         3.67702
21     22       

### 클래스별 AP 시각화

In [31]:
ap_per_class = metrics.box.ap50
colors = list(COLOR_MAP.values())

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(CLASSES, ap_per_class, color=colors, edgecolor="black", linewidth=0.8)
ax.axhline(y=float(np.mean(ap_per_class)), color="red",
           linestyle="--", linewidth=1.5, label=f"mAP = {np.mean(ap_per_class):.3f}")
ax.set_ylim(0, 1.05)
ax.set_ylabel("AP @ IoU=0.5")
ax.set_title("Per-Class AP@0.5 — PCB Defect Detection", fontsize=13, fontweight="bold")
ax.legend()
for bar, val in zip(bars, ap_per_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.3f}", ha="center", va="bottom", fontsize=9)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("eval_class_ap.png", dpi=150)
plt.show()
print('[SAVE] eval_class_ap.png')

<Figure size 1000x500 with 1 Axes>

[SAVE] eval_class_ap.png
